# 05 · Fine-tune B (RAW), C (RAW-MATCHED), D (CURATED) with AlignTune
**Purpose:** identical LoRA SFT for the three arms (AlignTune → Unsloth backend), then merge each adapter into the base (AlignTune `PEFTMerger`) so SGLang can serve it.
**Inputs:** `artifacts/train/{raw,raw_matched,curated}.jsonl`. **Outputs:** `artifacts/adapters/{B,C,D}/`, `artifacts/models/{B,C,D}_merged/`, `results/train_log_{B,C,D}.json`, `figures/loss_curves.png`.
**Runtime (T4):** roughly 1 min per 50 optimizer steps; B (RAW) is the longest.
Make sure no SGLang server is running (training and serving never share the GPU).

In [ ]:
import os, sys, json, gc
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/ps8/repo/src")
import ps8_common as C
C.ensure_env()
cfg = C.load_config(); FORCE = False
GPU, P = C.gpu_profile(cfg)
C.stop_server()
T, SEED = cfg["train"], cfg["seed"]
ARMS = {"B": "raw", "C": "raw_matched", "D": "curated"}
print("profile", GPU, "| hyperparameters (identical for all arms):", json.dumps(T))

In [ ]:
# Token-length check against max_seq_length (chat template applied exactly as in training).
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(cfg["student"])
for arm, name in ARMS.items():
    lens = [len(tok.apply_chat_template(json.loads(l)["messages"], tokenize=True)) for l in open(C.ART / "train" / f"{name}.jsonl")]
    lens.sort()
    print(f"{arm} {name}: n={len(lens)} tokens p50={lens[len(lens)//2]} max={lens[-1]} >{T['max_seq_length']}: {sum(l > T['max_seq_length'] for l in lens)}")

In [ ]:
import torch
from aligntune.core.backend_factory import create_sft_trainer
from aligntune.core.merge import PEFTMerger


def train_arm(arm, name):
    adapter, merged = C.ART / "adapters" / arm, C.ART / "models" / f"{arm}_merged"
    if (merged / "config.json").exists() and not FORCE:
        print(f"[{arm}] merged model exists, skipping"); return
    trainer = create_sft_trainer(
        model_name=cfg["student"], dataset_name=str(C.ART / "train" / f"{name}.jsonl"), backend="unsloth",
        output_dir=str(adapter), num_epochs=T["num_epochs"], batch_size=T["per_device_batch_size"],
        gradient_accumulation_steps=T["gradient_accumulation_steps"], learning_rate=T["learning_rate"],
        lr_scheduler=T["lr_scheduler"], warmup_ratio=T["warmup_ratio"], max_seq_length=T["max_seq_length"],
        use_peft=True, lora_r=T["lora_r"], lora_alpha=T["lora_alpha"], lora_dropout=T["lora_dropout"],
        lora_target_modules=T["lora_target_modules"], precision=P["precision"], seed=SEED, data_seed=SEED,
        report_to="none", logging_steps=5, eval_strategy="no", save_strategy="no",
    )
    out = trainer.train()
    log = trainer.trainer.state.log_history
    json.dump({"arm": arm, "dataset": name, "result": out, "log_history": log, "hyperparameters": T},
              open(C.RESULTS / f"train_log_{arm}.json", "w"), indent=1, default=str)
    print(f"[{arm}] {out}")
    del trainer; gc.collect(); torch.cuda.empty_cache()
    PEFTMerger().merge_lora(base_model=cfg["student"], adapter_path=str(adapter), output_path=str(merged),
                            torch_dtype="float16" if P["precision"] == "fp16" else "bfloat16")
    gc.collect(); torch.cuda.empty_cache()
    print(f"[{arm}] merged → {merged}")


for arm, name in ARMS.items():
    train_arm(arm, name)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
COLORS = {"B": "#eb6834", "C": "#1baf7a", "D": "#eda100"}
NAMES = {"B": "B · RAW", "C": "C · RAW-MATCHED", "D": "D · CURATED"}
fig, ax = plt.subplots(figsize=(8, 4), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
summary = {}
for arm in ARMS:
    lh = json.load(open(C.RESULTS / f"train_log_{arm}.json"))["log_history"]
    pts = [(h["epoch"], h["loss"]) for h in lh if "loss" in h]
    ax.plot(*zip(*pts), color=COLORS[arm], linewidth=2, label=NAMES[arm])
    summary[arm] = {"steps": max(h.get("step", 0) for h in lh), "final_logged_loss": pts[-1][1]}
ax.set_xlabel("epoch"); ax.set_ylabel("training loss (full sequence)")
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False); ax.grid(color="#e6e5e0", linewidth=0.8)
ax.set_title("Training loss — identical hyperparameters, different data", loc="left")
fig.tight_layout(); fig.savefig(C.FIGURES / "loss_curves.png", dpi=160); plt.close(fig)
print(json.dumps(summary, indent=1))
print("Saved:", sorted(p.name for p in (C.ART / "models").glob("*_merged")), C.FIGURES / "loss_curves.png")